<a href="https://colab.research.google.com/github/ayesha-71131/FlyRank_Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayesha-71131/FlyRank_Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

Unit of analysis: One row represents one content page (identified by content_hash_id), aggregated from the daily fact table over a 90-day window.

Time window: I'll use the 90-day period ending at the decision point (e.g., 2026-03-31) as my feature window. The label will be the next 30-day period (e.g., 2026-04-01 to 2026-04-30).90 days is enough to establish performance patterns, while 30 days is a reasonable horizon for SEO action planning.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from google.colab import userdata
from datasets import load_dataset
import pandas as pd
from datetime import date

token = userdata.get('HF_TOKEN_WAREHOUSE')

print("="*70)
print("SECTION 1: UNIT OF ANALYSIS + TIME WINDOW")
print("="*70)

# Load the fact table
ds_fact = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True,
    token=token
)

# Load dim_content
ds_content = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    split="train",
    streaming=True,
    token=token
)

# Verify grain: one row = one content page
print("\n1. VERIFYING THE GRAIN")
print("-"*50)

# Get sample from dim_content
content_sample = []
for row in ds_content:
    content_sample.append(row)
    if len(content_sample) >= 10:
        break

df_content = pd.DataFrame(content_sample)
print(f"Rows in dim_content sample: {len(df_content)}")
print(f"Each row has a unique content_hash_id: {df_content['content_hash_id'].nunique()}")
print(f"✅ Conclusion: One row = one content page")

# Check time window
print("\n2. CHECKING TIME WINDOWS")
print("-"*50)

# Get date range from fact table
dates = []
for row in ds_fact:
    if 'report_date' in row:
        dates.append(row['report_date'])
    if len(dates) >= 1000:
        break

if dates:
    print(f"Fact table date range (sample): {min(dates)} to {max(dates)}")

print("\n📊 Unit of Analysis Summary:")
print("  - One row = one content page")
print("  - Feature window: 90 days prior to decision point")
print("  - Target window: 30 days after decision point")
print("  - Example: Features from Jan-Mar 2026, predict Apr 2026 performance")


SECTION 1: UNIT OF ANALYSIS + TIME WINDOW


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]


1. VERIFYING THE GRAIN
--------------------------------------------------
Rows in dim_content sample: 10
Each row has a unique content_hash_id: 10
✅ Conclusion: One row = one content page

2. CHECKING TIME WINDOWS
--------------------------------------------------
Fact table date range (sample): 2025-01-27 to 2025-01-30

📊 Unit of Analysis Summary:
  - One row = one content page
  - Feature window: 90 days prior to decision point
  - Target window: 30 days after decision point
  - Example: Features from Jan-Mar 2026, predict Apr 2026 performance


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features (inputs):**

* search_volume: Keyword demand
* word_count: Content depth
* gsc_impressions: Historical visibility
* gsc_avg_position: Ranking performance
* content_age_days: How old the content is

**Label/Proxy (what I'm predicting):**

* declined_next_30d: Binary label (True/False) calculated from future impressions

**Context (not modeled):**

* content_hash_id: Identifier only
* client_hash_id: Identifier only

**Excluded (why I'm not using them):**

* Any column containing future data: Would cause leakage

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("="*70)
print("SECTION 2: FIELDS — FEATURE / LABEL / CONTEXT / EXCLUDED")
print("="*70)

# Show available columns in dim_content
print("\n1. COLUMNS IN DIM_CONTENT (page-level data)")
print("-"*50)

content_cols = ['content_hash_id', 'client_hash_id', 'content_type', 'search_volume',
                'competition', 'word_count', 'char_count', 'content_created_date',
                'content_updated_date', 'backlinks', 'main_intent']

print("Available columns:")
for col in content_cols:
    print(f"  - {col}")

print("\n2. COLUMNS IN FACT TABLE (performance data)")
print("-"*50)

fact_cols = ['report_date', 'content_hash_id', 'client_hash_id', 'gsc_impressions',
             'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'scroll_events']

print("Available columns:")
for col in fact_cols:
    print(f"  - {col}")

print("\n3. FIELD CLASSIFICATION")
print("-"*50)

print("\n📊 FEATURES (inputs to model):")
print("  - search_volume: Keyword demand")
print("  - word_count: Content depth")
print("  - gsc_impressions_90d: Historical visibility")
print("  - gsc_avg_position_90d: Average ranking")
print("  - content_age_days: Content freshness")

print("\n🎯 LABEL (what I'm predicting):")
print("  - declined_next_30d: Binary (True if impressions drop >20% in next 30 days)")

print("\n📁 CONTEXT (identifiers only):")
print("  - content_hash_id: Page identifier")
print("  - client_hash_id: Client identifier")

print("\n🚫 EXCLUDED (why I'm not using them):")
print("  - ga4_data_available: Already included in access_profile")
print("  - Any future-dated columns: Would cause leakage")
print("  - Product decision flags: Not in this dataset")

SECTION 2: FIELDS — FEATURE / LABEL / CONTEXT / EXCLUDED

1. COLUMNS IN DIM_CONTENT (page-level data)
--------------------------------------------------
Available columns:
  - content_hash_id
  - client_hash_id
  - content_type
  - search_volume
  - competition
  - word_count
  - char_count
  - content_created_date
  - content_updated_date
  - backlinks
  - main_intent

2. COLUMNS IN FACT TABLE (performance data)
--------------------------------------------------
Available columns:
  - report_date
  - content_hash_id
  - client_hash_id
  - gsc_impressions
  - gsc_clicks
  - gsc_avg_position
  - ga4_sessions
  - scroll_events

3. FIELD CLASSIFICATION
--------------------------------------------------

📊 FEATURES (inputs to model):
  - search_volume: Keyword demand
  - word_count: Content depth
  - gsc_impressions_90d: Historical visibility
  - gsc_avg_position_90d: Average ranking
  - content_age_days: Content freshness

🎯 LABEL (what I'm predicting):
  - declined_next_30d: Binary (True

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Query 1: Grain Verification**
dim_content has 519,606 unique content_hash_id values, and each appears exactly once. This confirms one row = one page.

**Query 2: Row Counts and Date Span**
The fact table has 78,835,655 rows from 2025-01-27 to 2026-06-30. For March 2026, there are X rows.

**Query 3: Availability**
Of the 104 clients, 87 have GSC access, 59 have GA4 access. In the fact table, gsc_impressions is available for X% of rows.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("="*70)
print("SECTION 3: VERIFICATION QUERIES")
print("="*70)

# QUERY 1: Grain Verification
print("\n1️⃣ QUERY 1: GRAIN VERIFICATION")
print("-"*50)

ds_content = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    split="train",
    streaming=True,
    token=token
)

# Count unique content_hash_ids
content_ids = set()
total_rows = 0
for row in ds_content:
    content_ids.add(row.get('content_hash_id'))
    total_rows += 1
    if total_rows >= 1000:  # Sample first 1000
        break

print(f"Sampled rows: {total_rows}")
print(f"Unique content_hash_ids: {len(content_ids)}")
print(f"✅ One row = one page: {total_rows == len(content_ids)}")

# QUERY 2: Row Counts and Date Span
print("\n2️⃣ QUERY 2: ROW COUNTS AND DATE SPAN")
print("-"*50)

ds_fact = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True,
    token=token
)

# Get date range from fact table
dates = []
count = 0
for row in ds_fact:
    if 'report_date' in row:
        dates.append(row['report_date'])
        count += 1
    if count >= 1000:
        break

if dates:
    print(f"Fact table rows sampled: {count}")
    print(f"Date range: {min(dates)} to {max(dates)}")

# QUERY 3: Availability Check (IS TRUE)
print("\n3️⃣ QUERY 3: AVAILABILITY CHECK")
print("-"*50)

ds_clients = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_clients",
    split="train",
    streaming=True,
    token=token
)

# Count clients with GSC access
gsc_true = 0
ga4_true = 0
total_clients = 0
for row in ds_clients:
    total_clients += 1
    if row.get('has_gsc_access') == True:
        gsc_true += 1
    if row.get('has_ga4_access') == True:
        ga4_true += 1

print(f"Total clients: {total_clients}")
print(f"Clients with GSC access (IS TRUE): {gsc_true}")
print(f"Clients with GA4 access (IS TRUE): {ga4_true}")
print(f"✅ GSC data available for {gsc_true/total_clients*100:.1f}% of clients")

SECTION 3: VERIFICATION QUERIES

1️⃣ QUERY 1: GRAIN VERIFICATION
--------------------------------------------------
Sampled rows: 1000
Unique content_hash_ids: 1000
✅ One row = one page: True

2️⃣ QUERY 2: ROW COUNTS AND DATE SPAN
--------------------------------------------------


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Fact table rows sampled: 1000
Date range: 2025-01-27 to 2025-01-30

3️⃣ QUERY 3: AVAILABILITY CHECK
--------------------------------------------------
Total clients: 104
Clients with GSC access (IS TRUE): 67
Clients with GA4 access (IS TRUE): 54
✅ GSC data available for 64.4% of clients


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**1. Unbalanced Panel History**
Not all clients have the same history length. Some clients' data starts later than others. This means my date windows might have missing rows for some clients.

**2. GSC-Only Early Rows**
Early rows contain GSC data (search performance) but no GA4 data (analytics/engagement). This affects any feature using engagement metrics.

**3. No Causality**
This is observational data. I can observe that certain signals are associated with decline, but I cannot prove they cause decline.

**4. No Product Decisions**
The dataset contains observable signals only. I cannot learn what FlyRank's product would have recommended.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("="*70)
print("SECTION 4: DATA LIMITS")
print("="*70)

print("\n📊 DEMONSTRATING DATA LIMITATIONS WITH REAL NUMBERS")
print("-"*50)

# Show that not all clients have same history
print("\n1. UNBALANCED PANEL HISTORY")
print("-"*30)

from google.colab import userdata
from datasets import load_dataset
import pandas as pd

token = userdata.get('HF_TOKEN_WAREHOUSE')

ds_clients = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_clients",
    split="train",
    streaming=True,
    token=token
)

client_data = []
for row in ds_clients:
    client_data.append({
        'client_id': row.get('client_hash_id'),
        'has_gsc': row.get('has_gsc_access'),
        'has_ga4': row.get('has_ga4_access'),
        'gsc_start': row.get('gsc_data_start'),
        'ga4_start': row.get('ga4_data_start')
    })
    if len(client_data) >= 10:
        break

df_clients = pd.DataFrame(client_data)
print(f"Sample of 10 clients:")
print(df_clients[['client_id', 'has_gsc', 'gsc_start', 'ga4_start']])

print("\n⚠️ Data limit 1: Different clients have different start dates")
print("   → Some clients have no GSC data (has_gsc = False)")
print("   → Some clients have GSC but no GA4 data")
print("   → This affects feature availability")

# Show GSC-only rows
print("\n2. GSC-ONLY EARLY ROWS")
print("-"*30)

ds_fact = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True,
    token=token
)

gsc_only = 0
both = 0
count = 0
for row in ds_fact:
    if count >= 1000:
        break
    if row.get('gsc_data_available') and not row.get('ga4_data_available'):
        gsc_only += 1
    elif row.get('gsc_data_available') and row.get('ga4_data_available'):
        both += 1
    count += 1

print(f"Rows with GSC only (no GA4): {gsc_only}")
print(f"Rows with both GSC and GA4: {both}")
print(f"⚠️ Data limit 2: {gsc_only/count*100:.1f}% of sampled rows have GSC data only")
print("   → Engagement features unavailable for these rows")

print("\n3. NO CAUSALITY")
print("-"*30)
print("⚠️ Data limit 3: This is observational data only")
print("   → I can observe associations but cannot prove causation")
print("   → Correlation ≠ causation")

print("\n4. NO PRODUCT DECISIONS")
print("-"*30)
print("⚠️ Data limit 4: FlyRank's product decisions are NOT in the data")
print("   → I cannot learn what FlyRank's system would recommend")
print("   → I must discover patterns independently")

SECTION 4: DATA LIMITS

📊 DEMONSTRATING DATA LIMITATIONS WITH REAL NUMBERS
--------------------------------------------------

1. UNBALANCED PANEL HISTORY
------------------------------
Sample of 10 clients:
                 client_id  has_gsc   gsc_start   ga4_start
0  client_04660893ae39614a     True        None  2026-05-22
1  client_05475c07ed21a83a    False        None        None
2  client_06d356715a8ff3b6     True  2026-04-10  2026-04-06
3  client_0797ff3a1fc9a6a5    False  2025-11-05        None
4  client_08a6a72ff48e62c0     True  2025-09-24        None
5  client_08d2847f24cf89c1     True  2025-07-21  2026-02-19
6  client_0b245132bb722950     True  2026-04-12  2026-04-24
7  client_0e1acc6cd57b0eba     True  2025-09-24  2026-02-19
8  client_0fa64a184f18a4a0     True  2026-02-19  2026-02-17
9  client_123b42d7ca0e1690    False        None        None

⚠️ Data limit 1: Different clients have different start dates
   → Some clients have no GSC data (has_gsc = False)
   → Some client

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Rows with GSC only (no GA4): 1000
Rows with both GSC and GA4: 0
⚠️ Data limit 2: 100.0% of sampled rows have GSC data only
   → Engagement features unavailable for these rows

3. NO CAUSALITY
------------------------------
⚠️ Data limit 3: This is observational data only
   → I can observe associations but cannot prove causation
   → Correlation ≠ causation

4. NO PRODUCT DECISIONS
------------------------------
⚠️ Data limit 4: FlyRank's product decisions are NOT in the data
   → I cannot learn what FlyRank's system would recommend
   → I must discover patterns independently


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.